In [1]:
# === SETUP (une seule fois) ===
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
from dotenv import load_dotenv

import sys
sys.path.append("../src")

from extraction_and_chunking import extract_and_chunk
from dep_index import index_chunks
from retrieve_chunks import retrieve_chunks
from build_promt import build_prompt
from generate_response import generate_answer

load_dotenv("../.env")

model = SentenceTransformer("all-MiniLM-L6-v2")
client_db = chromadb.PersistentClient(path="../vectorstore")
collection = client_db.get_or_create_collection(name="mon_cours")
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# === PIPELINE COMPLET ===
chunks, metadatas = extract_and_chunk("../data/cloud_course.pdf")
nb_indexed = index_chunks(chunks, metadatas, collection, model)
print(f"{nb_indexed} chunks indexés")

question = "quels sont les avantages du cloud computing ?"
retrieved_chunks, retrieved_metadatas = retrieve_chunks(question, collection, model, n_results=5)
prompt = build_prompt(question, retrieved_chunks, retrieved_metadatas)
answer = generate_answer(prompt, groq_client)

print(answer)

/home/mouad/anaconda3/envs/rag-assistant/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 787.05it/s]


270 chunks indexés
Les principaux avantages du cloud computing, selon le cours, sont :

* **Recentrage sur le cœur de métier** : les équipes informatiques sont déchargées, ce qui leur permet de se concentrer sur des activités à forte valeur ajoutée.  
* **Maintenance, sécurisation et évolutions gérées par le prestataire** : ces tâches sont prises en charge par le fournisseur, qui en fait son métier.  
* **Réduction des coûts** : diminution du coût total de possession des systèmes informatiques.  
* **Scalabilité facile** : possibilité d’augmenter ou de diminuer rapidement les ressources selon les besoins.  
* **Mutualisation des services** : économies réalisées grâce à la partagée des ressources entre de nombreux clients.

Source : pages 180 et 178.
